In [1]:
import os
import subprocess
import pandas as pd

In [2]:
in_dir = '../../data/encode_chipseq'
out_dir = '../../results/27_compare_tfbs_prediction/01_download_peaks'

In [ ]:
sample_list, tf_list = [], []

for sample in ['K562', 'HepG2']:
    # create output directory if not exists
    sample_out_dir = f"{out_dir}/{sample}"
    os.makedirs(sample_out_dir, exist_ok=True)

    df = pd.read_csv(f'{in_dir}/{sample}.txt', sep='\t', skiprows=1, header=None).astype(str)
    # df = df.head(1) # for testing

    # get url for each row and download if not exists
    for index, row in df.iterrows():
        url = row[0]
        filename = url.split("/")[-1].split(".")[0]
        out_file = f"{sample_out_dir}/{filename}.bed.gz"
        if not os.path.exists(out_file):
            subprocess.run(['wget', '-O', out_file, url, '--quiet'])

            # uncompress and sort the file
            subprocess.run(['gunzip', '-f', out_file])
            bed_file = out_file[:-3]
            sorted_bed_file = f"{sample_out_dir}/{filename}_sorted.bed"
            subprocess.run(['sort', '-k1,1', '-k2,2n', bed_file, '-o', sorted_bed_file])
            os.remove(bed_file)
            os.rename(sorted_bed_file, bed_file)

            sample_list.append(sample)
            tf_list.append(filename)

df_peaks = pd.DataFrame({'sample': sample_list, 'tf': tf_list})
df_peaks.to_csv(f"{out_dir}/tfbs_peaks_metadata.csv", index=False)